# EV Penetration Presentation



In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repository root:', REPO_ROOT)

## Overview

`run_ev_penetration` takes a base household load profile, selects houses on each feeder, adds EV charging profiles based on a seed, and gives the voltage and line performance.

## Inputs and Outputs

**Inputs**:
- `dataset`: power grid data including `sym_load` and network topology.
- `active_load_profiles`: time series of active power per house.
- `reactive_load_profiles`: matching reactive power per house.
- `ev_pool`: a set of EV charging profiles.
- `feeder_line_ids`: the top line IDs for each LV feeder.
- `graph_processor`: object with `find_downstream_vertices(feeder_line_id)`.
- `penetration_level`: fraction of houses with EVs (0.0 to 1.0).
- `random_seed`: used to make the assignment deterministic.

**Outputs**:
- `timestamp_table`: voltage summary per timestamp.
- `line_table`: summary per line, including losses and loading.

## Step 1: Validate inputs

The function first checks the `penetration_level`:
- must be a number
- must lie between `0.0` and `1.0`


## Step 2: Decide EV count per feeder

The number of EVs per feeder is calculated as:

`evs_per_feeder = floor(penetration_level * total_houses / number_of_feeders)`


## Step 3: Find downstream houses for each feeder

For each feeder line, the function uses `graph_processor.find_downstream_vertices(feeder_line_id)` to get all nodes on that feeder.

Then it maps those nodes to `sym_load` IDs so the EVs are assigned to actual houses.

## Step 4: Randomly select houses for EVs

Within each feeder:
- select `evs_per_feeder` houses from the downstream loads
- use the random seed for reproducibility
- use `replace=False` so a house is selected only once per feeder



## Step 5: Add EV charging to house profiles

For each selected house:
- pick an EV profile from `ev_pool`
- remove that profile from the available pool so it is not reused
- add the EV load to the house's active load profile

This creates an augmented active-load matrix containing both household and EV demand.

## Step 6: Run time-series power flow

Once the load profiles are updated, the function runs a time-series power flow.

The real implementation uses `PowerGridModel` to compute node voltages and line currents for every timestep.

## Step 7: Build summary tables

Two summary tables are created:

- `timestamp_table`: shows max/min voltage and the corresponding nodes per timestamp.
- `line_table`: shows total losses, max/min loading, and the timestamps when they occur.


In [ ]:

print('Starting EV penetration demo cell...')
import pandas as pd
import numpy as np


dataset = {
    'sym_load': pd.DataFrame({'id': [1, 2, 3, 4], 'node': [3, 4, 5, 6]})
}

timestamps = pd.date_range('2020-01-01', periods=4, freq='h')
active_load_profiles = pd.DataFrame(
    1.0,
    index=timestamps,
    columns=[str(i) for i in dataset['sym_load']['id']]
)
reactive_load_profiles = pd.DataFrame(
    0.1,
    index=timestamps,
    columns=active_load_profiles.columns
)
ev_pool = pd.DataFrame(
    {
        0: [0.0, 0.5, 0.2, 0.0],
        1: [0.1, 0.2, 0.0, 0.0],
        2: [0.0, 0.3, 0.1, 0.0],
        3: [0.2, 0.0, 0.0, 0.1],
    },
    index=timestamps,
)

class SimpleGraphProcessor:
    def find_downstream_vertices(self, feeder_line_id):
        if feeder_line_id == 10:
            return [3, 4]
        return [5, 6]

graph_processor = SimpleGraphProcessor()
feeder_line_ids = [10, 20]
penetration_level = 0.5

# Step 2: compute EVs per feeder
# Step 3: find downstream loads
# Step 4: select houses
# Step 5: add EV profiles to loads

total_houses = len(dataset['sym_load']['id'])
evs_per_feeder = int(np.floor(penetration_level * total_houses / len(feeder_line_ids)))
print('EVs per feeder:', evs_per_feeder)

rng = np.random.default_rng(42)
augmented_profiles = active_load_profiles.copy()
available_ev_profiles = list(range(len(ev_pool.columns)))

for feeder in feeder_line_ids:
    downstream_nodes = graph_processor.find_downstream_vertices(feeder)
    downstream_load_ids = dataset['sym_load'][dataset['sym_load']['node'].isin(downstream_nodes)]['id'].tolist()
    print('Feeder', feeder, 'downstream loads:', downstream_load_ids)

    selected_load_ids = rng.choice(downstream_load_ids, size=min(evs_per_feeder, len(downstream_load_ids)), replace=False)
    print('Selected loads for EVs:', selected_load_ids)

    for load_id in selected_load_ids:
        ev_profile_idx = rng.choice(len(available_ev_profiles))
        ev_profile_col = available_ev_profiles.pop(ev_profile_idx)
        augmented_profiles[str(load_id)] += ev_pool.iloc[:, ev_profile_col]
        print(f'  Assigned EV profile {ev_profile_col} to house {load_id}')

print('\nAugmented active load profiles:')
print(augmented_profiles)